In [0]:
import mlflow
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_databricks import ChatDatabricks, DatabricksVectorSearch

mlflow.set_registry_uri("databricks-uc")

PROMPT_VERSION = "v1-cited"
RETRIEVER_K = 5
LLM_ENDPOINT = "databricks-dbrx-instruct"
TEMPERATURE = 0.1

SYSTEM_PROMPT = """You are a pharmacovigilance assistant grounded in FDA data.

Rules:
1. Answer ONLY from the provided context. If insufficient, say "I don't have enough FDA data to answer this confidently."
2. Cite every claim using [drug_generic - section] format.
3. Never invent dosages, contraindications, or adverse events.
4. If asked about a drug not in context, say so explicitly.
5. End with: "This is not medical advice. Consult prescribing information."

Context:
{context}
"""

def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata.get('drug_generic','?')} - {d.metadata.get('section','?')}]\n{d.page_content}"
        for d in docs)

retriever = DatabricksVectorSearch(
    endpoint="fda-vs-endpoint",
    index_name="fda_rag.gold.fda_chunks_index",
    text_column="text",
    columns=["drug_generic", "drug_brand", "section"]
).as_retriever(search_kwargs={"k": RETRIEVER_K})

llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=TEMPERATURE)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("user", "{question}")])

chain = (
    {"context": retriever | RunnableLambda(format_docs),
     "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser())

# Required for log_model with code-based logging
mlflow.models.set_model(chain)

In [0]:

# LOG IT
import mlflow
mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="fda-rag-v1-cited"):
    mlflow.log_param("prompt_version", "v1-cited")
    mlflow.log_param("retriever_k", 5)
    mlflow.log_param("llm_endpoint", "databricks-dbrx-instruct")
    mlflow.log_param("temperature", 0.1)

    model_info = mlflow.langchain.log_model(
        lc_model="04_rag_chain.py",  # path to chain notebook
        artifact_path="chain",
        registered_model_name="fda_rag.gold.fda_assistant",
        input_example={"question": "What are cardiac adverse events for atorvastatin?"})

    print(f"Logged: {model_info.model_uri}")

In [0]:
# TEST
import mlflow
loaded = mlflow.langchain.load_model("models:/fda_rag.gold.fda_assistant/1")
print(loaded.invoke({"question": "What are common adverse reactions to metformin?"}))